# 00 — Datensatz-Partitionierung

Dieses Notebook erzeugt die drei disjunkten Datensatz-Splits, die für den gesamten Benchmark benötigt werden:

| Split | Zweck |
|---|---|
| **train** | NeuralFP-Training (Augmentation via MUSAN) |
| **ref** | Gemeinsame Referenz-Datenbank für alle drei Systeme |
| **ood_pool** | Out-of-Distribution-Queries (Song nicht in Ref-DB) |

Zusätzlich werden **Dry-Run-Subsets** (kleine Untermengen) für schnelle End-to-End-Tests angelegt.

**Abhängigkeiten:** Keine vorherigen Notebooks erforderlich.
**Ausgabe:** JSON-Dateien in `data/partitions/`


In [ ]:
# ── RUN MODE CONFIG ──────────────────────────────────────────────────────────
# Switch between dry run and live run by editing src/run_config.py:
#   RUN_MODE = "dry"   ← fast end-to-end test   (52 ref, 200 train, 10 OOD)
#   RUN_MODE = "live"  ← full benchmark run      (≈4930 ref, 10k train, 200 OOD)
# All notebooks read RUN_MODE from run_config.cfg automatically.
import sys
from pathlib import Path
_PROJECT_ROOT_CFG = Path("..").resolve()
sys.path.insert(0, str(_PROJECT_ROOT_CFG / "src"))
from run_config import cfg
RUN_MODE = cfg.run_mode
print(f"RUN_MODE = {RUN_MODE!r}")

## 1. Imports und Pfade

Projekt-Root wird als übergeordnetes Verzeichnis von `notebooks/` bestimmt,
damit alle relativen Pfade unabhängig vom Start-Verzeichnis funktionieren.
`SEED = 42` gilt für alle Zufallsprozesse in diesem Notebook.


In [ ]:
import os, sys, json
from pathlib import Path
import numpy as np

# Notebook lives in notebooks/; project root is one level up.
PROJECT_ROOT = Path("..").resolve()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from run_config import cfg
RUN_MODE = cfg.run_mode

from utils import (
    load_fma_metadata, create_partitions, create_dry_run_subsets,
    split_musan, filter_and_replenish_by_duration,
    assert_disjoint, print_genre_distribution, print_missing_files,
    save_partitions,
)

FMA_DIR        = PROJECT_ROOT / "data" / "fma_medium"
MUSAN_DIR      = PROJECT_ROOT / "data" / "musan"
PARTITIONS_DIR = PROJECT_ROOT / "data" / "partitions"
SEED = 42
print(f"RUN_MODE = {RUN_MODE!r}")
print(f"Targets — ref: {cfg.n_ref}, train: {cfg.n_train}, ood: {cfg.n_ood}, query: {cfg.n_query}")

## 2. FMA-Metadaten laden

Liest `fma_metadata/tracks.csv` (mehrstufiger Header), filtert auf das
`medium`-Subset und prüft, ob die .mp3-Datei jedes Tracks tatsächlich auf
dem Dateisystem vorhanden ist. Tracks ohne Audiodatei werden ausgeschlossen.

Ergebnis: DataFrame mit Spalten `genre`, `duration`, `filepath`, Index = `track_id`.


In [ ]:
track_df = load_fma_metadata(FMA_DIR)
print(f"Loaded: {len(track_df)} tracks")
print(track_df.head())


## 3. Dauer-Filter (≥ 30 s)

Tracks kürzer als 30 Sekunden werden ausgeschlossen, da sie zu kurz für
zuverlässige 10-Sekunden-Query-Segmente mit Puffer sind.


In [ ]:
before = len(track_df)
track_df = track_df[track_df["duration"] >= 30.0].copy()
print(f"Duration filter (>=30s): {before} → {len(track_df)} tracks "
      f"({before - len(track_df)} removed)")


## 4. Vollständige Partitionierung — Train / Ref / OOD-Pool

Genre-stratifiziertes Sampling in zwei Schritten:
1. `train` aus dem Gesamt-Pool ziehen
2. `ref` aus den verbleibenden Tracks ziehen
3. Rest → `ood_pool`

Da nur ~17 000 der 25 000 fma_medium-Tracks heruntergeladen sind,
werden die Zielgrößen proportional skaliert (~59 % / ~29 % / ~12 %).
`assert_disjoint` wird intern nach jedem Sampling aufgerufen.


In [ ]:
# fma_medium has ~25k entries but only ~17k audio files are downloaded.
# Scale partition sizes proportionally to what's available.
N_AVAIL = len(track_df)
N_TRAIN = cfg.n_train   # 15000
N_REF   = cfg.n_ref     # 8000
print(f"Available: {N_AVAIL}  →  n_train={N_TRAIN}, n_ref={N_REF}")

partitions = create_partitions(
    track_df, n_train=N_TRAIN, n_ref=N_REF, seed=SEED
)
print("Full partition sizes:", {k: len(v) for k, v in partitions.items()})


## 5. Dry-Run-Subsets ziehen (roh)

Kleine genre-proportionale Stichproben aus jedem der drei Splits:
- `dry_ref`:   ~50 Tracks  → Referenz-DB für schnellen Test
- `dry_train`: 200 Tracks  → NeuralFP-Training (2 Epochen)
- `dry_ood`:    10 Tracks  → OOD-Queries

Die Subsets werden direkt aus den Eltern-Splits entnommen und sind
per Konstruktion deren echte Teilmengen.


In [ ]:
if RUN_MODE == "dry":
    dry_raw = create_dry_run_subsets(
        ref_ids      = partitions["ref"],
        train_ids    = partitions["train"],
        ood_pool_ids = partitions["ood_pool"],
        metadata_df  = track_df,
        n_ref=cfg.n_ref, n_train=cfg.n_train, n_ood=cfg.n_ood, seed=SEED,
    )
else:
    print("Live run — skipping dry-run subset creation.")

In [ ]:
if RUN_MODE == "live":
    # Live run: use all of ref and train; draw 200 from ood_pool.
    # Target n_ref=8000 exceeds available ref pool (≈4930), so we cap at actual size.
    N_REF_LIVE   = min(cfg.n_ref,   len(partitions["ref"]))
    N_TRAIN_LIVE = len(partitions["train"])        # use all train tracks
    N_OOD_LIVE   = min(cfg.n_ood,   len(partitions["ood_pool"]))
    N_QUERY_LIVE = min(cfg.n_query, N_REF_LIVE)

    print(f"Live targets  : ref={cfg.n_ref}, train={cfg.n_train}, ood={cfg.n_ood}, query={cfg.n_query}")
    print(f"Available     : ref={len(partitions['ref'])}, train={len(partitions['train'])}, "
          f"ood_pool={len(partitions['ood_pool'])}")
    print(f"Actual        : ref={N_REF_LIVE}, train={N_TRAIN_LIVE}, ood={N_OOD_LIVE}, query={N_QUERY_LIVE}")

    _live_raw = create_dry_run_subsets(
        ref_ids      = partitions["ref"],
        train_ids    = partitions["train"],
        ood_pool_ids = partitions["ood_pool"],
        metadata_df  = track_df,
        n_ref=N_REF_LIVE, n_train=N_TRAIN_LIVE, n_ood=N_OOD_LIVE, seed=SEED,
    )
    live_ref_ids   = _live_raw["dry_ref"]    # ≈4930 tracks
    live_train_ids = _live_raw["dry_train"]  # ≈10000 tracks
    live_ood_ids   = _live_raw["dry_ood"]    # 200 tracks

    # live_query: random 500 from live_ref (query songs — subset of ref DB)
    _rng_q = np.random.default_rng(SEED + 10)
    live_query_ids = sorted(
        _rng_q.choice(live_ref_ids, size=N_QUERY_LIVE, replace=False).tolist()
    )
    assert set(live_query_ids) <= set(live_ref_ids), "live_query ⊄ live_ref!"
    print(f"  live_query: {len(live_query_ids)} tracks (subset of live_ref) ✓")

## 6. Dauer-Prüfung der Dry-Run-Subsets und Auffüllen

Auch innerhalb der bereits gefilterten Partition können vereinzelt Tracks
unter 30 s liegen (z. B. wenn sich Metadaten-Dauer und tatsächliche Datei
unterscheiden). Diese werden durch zufällige Draws aus dem Rest der
jeweiligen Eltern-Partition ersetzt.


In [ ]:
if RUN_MODE == "dry":
    # Build per-split pools (disjoint from each split)
    ref_pool   = [t for t in partitions["ref"]      if t not in dry_raw["dry_ref"]]
    train_pool = [t for t in partitions["train"]    if t not in dry_raw["dry_train"]]
    ood_pool   = [t for t in partitions["ood_pool"] if t not in dry_raw["dry_ood"]]

    dry_ref   = filter_and_replenish_by_duration(dry_raw["dry_ref"],   ref_pool,   track_df, 30.0, SEED)
    dry_train = filter_and_replenish_by_duration(dry_raw["dry_train"], train_pool, track_df, 30.0, SEED+1)
    dry_ood   = filter_and_replenish_by_duration(dry_raw["dry_ood"],   ood_pool,   track_df, 30.0, SEED+2)
else:
    # Duration already enforced on full partitions (step 3).
    # Replenish only live_ood which is a subset of ood_pool.
    _ood_remaining = [t for t in partitions["ood_pool"] if t not in live_ood_ids]
    live_ood_ids = filter_and_replenish_by_duration(live_ood_ids, _ood_remaining, track_df, 30.0, SEED+2)
    print(f"Live: ref={len(live_ref_ids)}, train={len(live_train_ids)}, "
          f"ood={len(live_ood_ids)}, query={len(live_query_ids)}")

## 7. MUSAN-Split (80 % Train / 20 % Eval)

MUSAN-Dateien aus den Kategorien `speech` und `noise` (nicht `music`)
werden deterministisch gemischt und aufgeteilt:
- **80 % train**: Noise-Augmentation beim NeuralFP-Training
- **20 % eval**:  Noise-Distortionen für Query-Generierung in NB 01

Die beiden Hälften sind strikt disjunkt — kein Data Leakage durch Noise.


In [ ]:
musan_split = split_musan(MUSAN_DIR, split=0.8, seed=SEED)
print(f"MUSAN train: {len(musan_split['train'])} files")
print(f"MUSAN eval:  {len(musan_split['eval'])}  files")


## 8. Data-Leakage-Check: Disjunktheit

Kritischste Validierung des gesamten Benchmarks. Drei Prüfungen:
1. Vollständige Splits (`train`, `ref`, `ood_pool`) sind paarweise disjunkt
2. Dry-Run-Subsets (`dry_ref`, `dry_train`, `dry_ood`) sind paarweise disjunkt
3. Jedes Dry-Subset ist echte Teilmenge seines Eltern-Splits

OOD-Songs dürfen **niemals** in der Referenz-DB oder im Training auftauchen —
andernfalls wäre die Specificity-Metrik wertlos.


In [ ]:
if RUN_MODE == "dry":
    print("=== Full partitions ===")
    assert_disjoint(partitions["train"], partitions["ref"], partitions["ood_pool"])

    print("\n=== Dry subsets ===")
    assert_disjoint(dry_ref, dry_train, dry_ood)

    print("\n=== dry_ref ⊂ ref, dry_train ⊂ train, dry_ood ⊂ ood_pool ===")
    assert set(dry_ref)   <= set(partitions["ref"]),      "dry_ref not subset of ref!"
    assert set(dry_train) <= set(partitions["train"]),    "dry_train not subset of train!"
    assert set(dry_ood)   <= set(partitions["ood_pool"]), "dry_ood not subset of ood_pool!"
    print("Subset checks passed. ✓")
else:
    print("=== Full partitions ===")
    assert_disjoint(partitions["train"], partitions["ref"], partitions["ood_pool"])

    print("\n=== Live subsets ===")
    assert_disjoint(live_ref_ids, live_train_ids, live_ood_ids)

    print("\n=== live_ref ⊂ ref, live_train ⊂ train, live_ood ⊂ ood_pool, live_query ⊂ live_ref ===")
    assert set(live_ref_ids)   <= set(partitions["ref"]),      "live_ref not subset of ref!"
    assert set(live_train_ids) <= set(partitions["train"]),    "live_train not subset of train!"
    assert set(live_ood_ids)   <= set(partitions["ood_pool"]), "live_ood not subset of ood_pool!"
    assert set(live_query_ids) <= set(live_ref_ids),           "live_query not subset of live_ref!"
    print("Subset checks passed. ✓")

## 9. Genre-Verteilungen der Dry-Run-Subsets

Kontrolliert, ob die genre-stratifizierte Stichprobe die Verteilung des
Gesamt-Datensatzes (Rock ~35 %, Electronic ~30 %, ...) reproduziert.
Starke Abweichungen würden die Vergleichbarkeit der Systeme beeinträchtigen.


In [ ]:
if RUN_MODE == "dry":
    print("--- dry_ref ---")
    print_genre_distribution(dry_ref, track_df)
    print("\n--- dry_train ---")
    print_genre_distribution(dry_train, track_df)
    print("\n--- dry_ood ---")
    print_genre_distribution(dry_ood, track_df)
else:
    print("--- live_ref (first 200 shown) ---")
    print_genre_distribution(live_ref_ids[:200], track_df)
    print("\n--- live_ood ---")
    print_genre_distribution(live_ood_ids, track_df)
    print("\n--- live_query ---")
    print_genre_distribution(live_query_ids, track_df)

## 10. Fehlende Audio-Dateien prüfen

Stellt sicher, dass alle in die Dry-Run-Subsets aufgenommenen Tracks
tatsächlich als .mp3-Dateien auf dem Dateisystem vorliegen.
Fehlende Dateien würden in NB 01 zu stillen Fehlern führen.


In [ ]:
if RUN_MODE == "dry":
    print("--- Missing files in dry_ref ---")
    print_missing_files(dry_ref, track_df)
    print("--- Missing files in dry_train ---")
    print_missing_files(dry_train, track_df)
    print("--- Missing files in dry_ood ---")
    print_missing_files(dry_ood, track_df)
else:
    print("--- Missing files in live_ref (sample: first 200) ---")
    print_missing_files(live_ref_ids[:200], track_df)
    print("--- Missing files in live_ood ---")
    print_missing_files(live_ood_ids, track_df)
    print("--- Missing files in live_query ---")
    print_missing_files(live_query_ids, track_df)

## 11. Partitionen persistieren

Alle Splits werden als JSON-Arrays in `data/partitions/` gespeichert.
Der MUSAN-Split wird separat als `musan_split.json` abgelegt
(enthält absolute Dateipfade statt Track-IDs).

Diese Dateien sind die einzige Datenquelle für alle nachfolgenden Notebooks —
**nie manuell bearbeiten**.


In [ ]:
# Always save full partitions (shared by dry and live runs)
base_partitions = {
    "train":    partitions["train"],
    "ref":      partitions["ref"],
    "ood_pool": partitions["ood_pool"],
}

if RUN_MODE == "dry":
    mode_partitions = {
        "dry_ref":   dry_ref,
        "dry_train": dry_train,
        "dry_ood":   dry_ood,
    }
else:
    mode_partitions = {
        "live_ref":   live_ref_ids,
        "live_train": live_train_ids,
        "live_ood":   live_ood_ids,
        "live_query": live_query_ids,
    }

all_partitions = {**base_partitions, **mode_partitions}
save_partitions(all_partitions, PARTITIONS_DIR)

# Save MUSAN split separately (same for both modes)
musan_json = PARTITIONS_DIR / "musan_split.json"
with open(musan_json, "w") as fh:
    json.dump(musan_split, fh)
print(f"Saved musan_split.json → {musan_json}")

## 12. Abschließender Überblick (Sanity Check)

Zusammenfassung aller erzeugten Splits. Wird am Ende jedes Notebooks
ausgegeben, um bei späterem Nachvollziehen einen schnellen Überblick zu
ermöglichen.


In [ ]:
import pandas as pd
print("=== Sanity check ===")
print(f"train    : {len(partitions['train']):>6} tracks")
print(f"ref      : {len(partitions['ref']):>6} tracks")
print(f"ood_pool : {len(partitions['ood_pool']):>6} tracks")

if RUN_MODE == "dry":
    print(f"dry_ref  : {len(dry_ref):>6} tracks")
    print(f"dry_train: {len(dry_train):>6} tracks")
    print(f"dry_ood  : {len(dry_ood):>6} tracks")
    summary = pd.DataFrame({
        "set": ["train", "ref", "ood_pool", "dry_ref", "dry_train", "dry_ood"],
        "n":   [len(partitions["train"]), len(partitions["ref"]),
                len(partitions["ood_pool"]),
                len(dry_ref), len(dry_train), len(dry_ood)],
    })
else:
    print(f"live_ref  : {len(live_ref_ids):>6} tracks  (target: {cfg.n_ref})")
    print(f"live_train: {len(live_train_ids):>6} tracks  (target: {cfg.n_train})")
    print(f"live_ood  : {len(live_ood_ids):>6} tracks")
    print(f"live_query: {len(live_query_ids):>6} tracks")
    summary = pd.DataFrame({
        "set": ["train", "ref", "ood_pool", "live_ref", "live_train", "live_ood", "live_query"],
        "n":   [len(partitions["train"]), len(partitions["ref"]),
                len(partitions["ood_pool"]),
                len(live_ref_ids), len(live_train_ids), len(live_ood_ids), len(live_query_ids)],
    })
print(summary)